# Foundation models on the 2025-26 RTM window (Kaggle GPU)

Zero-shot forecasts with Chronos-Bolt and TimesFM for the same 150 days, folds and scores as the CPU
ladder. For each forecast day D the model reads the filled RTM price up to the end of day D - 1 and
forecasts the 96 blocks of D. Neither model sees DAM prices, demand or any other input: the day-ahead
price is the strongest input of the ladder models, so these two are a univariate reference, not a
like-for-like contest.

**Input:** a private Kaggle dataset holding `rtm_dam_prices.parquet`, written by
`scripts/export_kaggle_inputs.py`. It has RTM and DAM prices and fold labels only.

**Output:** `predictions_chronos_bolt.parquet` and `predictions_timesfm.parquet` in `/kaggle/working`,
in the format of the ladder. Copy them into `.project-local/results_window/` and run the report stage.

**Status:** written without a GPU. The model calls follow the public descriptions of the two libraries
and have not been run yet. The first run decides whether they need changing.

In [ ]:
%pip install -q chronos-forecasting "timesfm[torch]"

In [ ]:
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

CONTEXT = 2016  # blocks of history per forecast: 21 days
HORIZON = 96
BATCH = 16
CAP = 10000.0
CAP_SHARE = 0.95
OUT = Path("/kaggle/working")
RUN = ["chronos_bolt", "timesfm"]
CHRONOS_ID = "amazon/chronos-bolt-base"
TIMESFM_ID = "google/timesfm-2.5-200m-pytorch"

In [ ]:
found = sorted(Path("/kaggle/input").rglob("rtm_dam_prices.parquet")) or [
    Path("rtm_dam_prices.parquet")
]
data = pd.read_parquet(found[0]).sort_values(["date", "block"]).reset_index(drop=True)
history = data["rtm_price_filled"].to_numpy(dtype=float)
start = data.groupby("date").head(1).set_index("date").index
first_row = {day: int(i) for i, day in zip(data.groupby("date").head(1).index, start, strict=True)}
days = sorted(data.loc[data["fold"].notna(), "date"].unique())
print(found[0], len(data), "rows,", len(days), "forecast days")

In [ ]:
def contexts_for(batch):
    return [history[max(0, first_row[day] - CONTEXT) : first_row[day]] for day in batch]


def forecast_days(predict, name):
    """Forecast every fold day. ``predict`` takes a list of arrays and returns (days, 96)."""
    parts = []
    started = time.time()
    for first in range(0, len(days), BATCH):
        batch = days[first : first + BATCH]
        values = np.asarray(predict(contexts_for(batch)), dtype=float)
        if values.shape != (len(batch), HORIZON):
            raise ValueError(f"{name}: got {values.shape}, expected {(len(batch), HORIZON)}")
        for day, row in zip(batch, values, strict=True):
            rows = data[data["date"] == day]
            part = rows[["date", "block", "rtm_price", "fold", "period"]].rename(
                columns={"rtm_price": "actual"}
            )
            part["prediction"] = row[part["block"].to_numpy() - 1]
            part["model"] = name
            parts.append(part)
    print(f"{name}: {time.time() - started:.0f}s")
    return pd.concat(parts, ignore_index=True)


def yesterday(contexts):
    return np.stack([c[-HORIZON:] for c in contexts])

In [ ]:
results = {"naive_yesterday_check": forecast_days(yesterday, "naive_yesterday_check")}

In [ ]:
if "chronos_bolt" in RUN:
    try:
        import torch
        from chronos import BaseChronosPipeline

        pipeline = BaseChronosPipeline.from_pretrained(
            CHRONOS_ID,
            device_map="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.bfloat16,
        )

        def chronos_predict(contexts):
            tensors = [torch.tensor(c, dtype=torch.float32) for c in contexts]
            forecast = np.asarray(pipeline.predict(tensors, prediction_length=HORIZON).float())
            return forecast[:, forecast.shape[1] // 2, :]  # the median quantile

        results["chronos_bolt"] = forecast_days(chronos_predict, "chronos_bolt")
    except Exception:
        traceback.print_exc()

In [ ]:
if "timesfm" in RUN:
    try:
        import timesfm

        model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_ID)
        model.compile(
            timesfm.ForecastConfig(
                max_context=CONTEXT,
                max_horizon=HORIZON,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
            )
        )

        def timesfm_predict(contexts):
            point, _ = model.forecast(horizon=HORIZON, inputs=contexts)
            return np.asarray(point)

        results["timesfm"] = forecast_days(timesfm_predict, "timesfm")
    except Exception:
        traceback.print_exc()

In [ ]:
def scores(frame):
    rows = {}
    for name, part in frame[frame["actual"].notna()].groupby("model"):
        a, p = part["actual"].to_numpy(), part["prediction"].to_numpy()
        at_cap, called = a >= CAP, p >= CAP_SHARE * CAP
        hits = int((at_cap & called).sum())
        prec = hits / max(int(called.sum()), 1)
        rec = hits / max(int(at_cap.sum()), 1)
        rows[name] = {
            "MAE": np.mean(np.abs(a - p)),
            "RMSE": np.sqrt(np.mean((a - p) ** 2)),
            "sMAPE": np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))) * 100,
            "cap_f1": 2 * prec * rec / (prec + rec) if prec + rec > 0 else float("nan"),
        }
    return pd.DataFrame(rows).T


everything = pd.concat(results.values(), ignore_index=True)
for label, part in [
    ("test", everything[everything["period"] == "test"]),
    ("validation", everything[everything["period"] == "validation"]),
    *[(f"fold {k}", everything[everything["fold"] == k]) for k in range(1, 6)],
]:
    print(f"== {label} ==")
    print(scores(part).round(2).to_string())

In [ ]:
for name, frame in results.items():
    if name != "naive_yesterday_check":
        frame.assign(fold=frame["fold"].astype(int)).to_parquet(
            OUT / f"predictions_{name}.parquet", index=False
        )
        print("wrote", OUT / f"predictions_{name}.parquet")